# Bloco 4 — Modelo de Otimização de Escalas

Este notebook implementa o modelo de otimização de escalas para o departamento de Housekeeping, com evolução progressiva da formulação.

## Organização do notebook

| Secções | Âmbito |
|---|---|
| 1–14 | Modelo base: recursos internos, T1 e T2, 1 semana |
| 15–18 | Extensão híbrida: recursos externos + análise de cenários |
| 19–21 | Análise de sensibilidade aos pesos da função objetivo |
| 5b | Elegibilidade funcional hierárquica |

## Como usar

- **Correr só o modelo base:** executar as secções 1–14
- **Correr o notebook completo:** Run All
- **Mudar a semana:** alterar `SEMANA_PREV_INICIO` / `SEMANA_PREV_FIM` no Bloco 3


## 1. Importações

Carregamento das bibliotecas necessárias. **PuLP** é a biblioteca de otimização que permite definir problemas ILP em Python e chamar solvers externos como o CBC.

In [45]:
# Instalar PuLP se ainda não estiver instalado (descomentar na primeira execução)
# !pip install pulp

from pathlib import Path   # manipulação de caminhos de ficheiros
import pandas as pd        # manipulação de dados tabulares
import numpy as np         # operações numéricas
import time                # medição do tempo de execução

# PuLP — biblioteca de otimização ILP
# LpProblem   : define o problema de minimização/maximização
# LpMinimize  : indica que queremos minimizar a função objetivo
# LpVariable  : cria variáveis de decisão (binárias, contínuas, etc.)
# lpSum       : soma eficiente de variáveis PuLP (equivalente ao Σ matemático)
# LpStatus    : converte o código de status do solver em texto legível
# value       : extrai o valor numérico de uma variável após resolução
# PULP_CBC_CMD: interface para o solver CBC (open-source, incluído com PuLP)
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus, value, PULP_CBC_CMD

print("PuLP importado com sucesso.")

PuLP importado com sucesso.


## 2. Carregar dados

São carregados 6 dos 10 ficheiros CSV do dataset — os suficientes para o modelo de otimização.
Os restantes (atividade, tipologias, atividade_tipologia, recursos_externos) são usados nos Blocos 2 e 3.

| Ficheiro | Uso no modelo |
|---|---|
| colaboradores | custos, limites contratuais, flags de elegibilidade |
| turnos | duração de cada turno |
| cobertura_necessidades | N_min e N_ideal por dia/turno/função |
| disponibilidade | saber quem está disponível em cada dia/turno |
| qualificacoes | saber quem pode executar cada função |
| preferencias | scores de preferência de turno e dia de folga |

In [46]:
ROOT_DIR         = Path.cwd().parent
DATA_DIR         = ROOT_DIR / "data" / "synthetic"
DATA_DIR_OUTPUTS = ROOT_DIR / "outputs"

# Carregar os 6 ficheiros necessários para o modelo de otimização
colab   = pd.read_csv(DATA_DIR / "colaboradores.csv")
turnos  = pd.read_csv(DATA_DIR / "turnos.csv")
cob     = pd.read_csv(DATA_DIR / "cobertura_necessidades.csv")
disp    = pd.read_csv(DATA_DIR / "disponibilidade.csv")
qual    = pd.read_csv(DATA_DIR / "qualificacoes.csv")
pref    = pd.read_csv(DATA_DIR / "preferencias.csv")

# Converter colunas de data para o tipo datetime do pandas
# (necessário para filtrar por intervalo de datas nas células seguintes)
cob["data"]  = pd.to_datetime(cob["data"])
disp["data"] = pd.to_datetime(disp["data"])

# Verificação: confirmar que os dados foram carregados e os ranges de datas são os esperados
print("Dados carregados:")
for nome, df in [("colaboradores",colab),("turnos",turnos),("cobertura",cob),
                  ("disponibilidade",disp),("qualificacoes",qual),("preferencias",pref)]:
    print(f"  {nome}: {df.shape[0]} linhas x {df.shape[1]} colunas")
print(f"\nRange datas cobertura    : {cob['data'].min().date()} a {cob['data'].max().date()}")
print(f"Range datas disponibilidade: {disp['data'].min().date()} a {disp['data'].max().date()}")

Dados carregados:
  colaboradores: 20 linhas x 19 colunas
  turnos: 3 linhas x 13 colunas
  cobertura: 3285 linhas x 8 colunas
  disponibilidade: 20075 linhas x 12 colunas
  qualificacoes: 75 linhas x 8 colunas
  preferencias: 60 linhas x 5 colunas

Range datas cobertura    : 2025-01-01 a 2025-12-31
Range datas disponibilidade: 2025-01-01 a 2025-12-31


## 3. Conjuntos e horizonte de planeamento

O modelo matemático é indexado por 4 conjuntos principais, que correspondem às dimensões do problema:

| Símbolo | Nome | Descrição |
|---|---|---|
| **C** | Colaboradores | Lista de IDs dos colaboradores activos |
| **D** | Dias | Lista de datas da semana em formato string (ex: "2025-01-01") |
| **T** | Turnos | ["T1", "T2"] — T3 excluído do âmbito do projecto |
| **F** | Funções | ["Auxiliar de limpeza", "Empregada de andares", "Supervisora"] |

A variável de decisão $x_{c,d,t,f}$ terá um valor por cada combinação possível destes 4 índices.

In [47]:
import joblib

# ── Semana-alvo: lida do gb_config.pkl gerado pelo Bloco 3 ───────────────────
# Alterar SEMANA_PREV_INICIO / SEMANA_PREV_FIM no Bloco 3 para mudar a semana
# Não alterar aqui — as datas propagam automaticamente via gb_config.pkl
config        = joblib.load(DATA_DIR_OUTPUTS / "gb_config.pkl")
SEMANA_INICIO = config["semana_inicio"]
SEMANA_FIM    = config["semana_fim"]

# Gerar lista de datas (disponível para todas as células seguintes)
datas = pd.date_range(SEMANA_INICIO, SEMANA_FIM)
D     = [d.strftime("%Y-%m-%d") for d in datas]

# Conjuntos base do modelo
T = ["T1", "T2"]
C = sorted(colab[colab["ativo"] == 1]["colaborador_id"].unique())
F = sorted(cob[cob["turno_id"].isin(T)]["funcao"].unique())

print(f"Semana-alvo : {SEMANA_INICIO} a {SEMANA_FIM}")
print(f"C = {len(C)} colaboradores | D = {len(D)} dias | T = {T}")
print(f"F = {F}")

Semana-alvo : 2025-01-01 a 2025-01-07
C = 20 colaboradores | D = 7 dias | T = ['T1', 'T2']
F = ['Auxiliar de limpeza', 'Empregada de andares', 'Supervisora']


## 4. Parâmetros escalares

Os parâmetros são valores fixos que o modelo usa nas restrições e na função objetivo. São construídos como dicionários Python — estrutura eficiente para lookups por chave.

| Parâmetro | Tipo de chave | Significado |
|---|---|---|
| H_t | turno_id | Duração em horas de cada turno |
| H_c | colaborador_id | Máximo de horas/dia por contrato |
| W_c | colaborador_id | Máximo de dias/semana por contrato |
| M_c | colaborador_id | Máximo de dias consecutivos |
| Cost_c | colaborador_id | Custo por hora (€/h) |

In [48]:
# H_t: duração de cada turno em horas
# Usado em H5 (limite horas diárias) e na função objetivo (custo = cost_c × H_t × x)
H_t = dict(zip(turnos["turno_id"], turnos["duracao_horas"]))

# H_c: horas máximas diárias de cada colaborador (definidas no contrato)
# Usado em H5: garante que nenhum colaborador excede o seu limite diário
H_c = dict(zip(colab["colaborador_id"], colab["horas_max_dia"]))

# W_c: dias máximos de trabalho por semana de cada colaborador
# Usado em H6: limita o número total de dias trabalhados na semana
W_c = dict(zip(colab["colaborador_id"], colab["dias_max_trabalho_semana"]))

# M_c: máximo de dias consecutivos permitidos (tipicamente 4-5 para FT, 2-3 para PT)
# Usado em H10: impede que o mesmo colaborador trabalhe demasiados dias seguidos
M_c = dict(zip(colab["colaborador_id"], colab["max_dias_consecutivos"]))

# Cost_c: custo por hora de cada colaborador (varia com função e experiência)
# Usado na função objetivo: minimizar custo total = Σ Cost_c × H_t × x_{c,d,t,f}
Cost_c = dict(zip(colab["colaborador_id"], colab["custo_hora"]))

print("H_t (horas por turno):", {t: H_t[t] for t in T})
print("Custo/hora (amostra):", dict(list(Cost_c.items())[:4]))

H_t (horas por turno): {'T1': 7, 'T2': 8}
Custo/hora (amostra): {'C001': 13.05, 'C002': 8.92, 'C003': 6.8, 'C004': 6.28}


## 5. Qualificações e conjuntos Cf

Para saber se um colaborador pode ser alocado a uma função, é necessário cruzar a tabela de qualificações com as funções do modelo.

O mapeamento é necessário porque a tabela de qualificações usa nomes técnicos (ex: `quartos_saida`) enquanto o modelo usa os nomes das funções (ex: `Empregada de andares`).

**Cf[f]** é o subconjunto de colaboradores elegíveis para a função f — usado na restrição H1 para garantir que só colaboradores elegíveis contam para a cobertura mínima.

> A célula 5b (imediatamente a seguir) aplica a elegibilidade funcional hierárquica, refinando `Cf[f]` com base no perfil contratual de cada colaborador. A célula 5 constrói `Qual` (qualificação técnica); a célula 5b constrói `Elegivel` (qualificação + hierarquia) e recalcula `Cf`.

In [49]:
# Mapeamento entre o nome da função (usado no modelo) e o código de qualificação (usado na tabela)
# Este mapeamento é específico ao projecto e foi definido na fase de preparação dos dados (Bloco 2)
map_funcao_qual = {
    "Empregada de andares" : "quartos_saida",    # limpeza de quartos com check-out
    "Supervisora"          : "supervisao",        # supervisão de equipa e qualidade
    "Auxiliar de limpeza"  : "areas_publicas",   # limpeza de corredores, lobbies, etc.
}

# Construir o conjunto de pares (colaborador, qualificação) onde pode_executar=1
# ou seja: todos os colaboradores que têm permissão para executar uma qualificação
qual_set = set(
    (r["colaborador_id"], r["qualificacao"])
    for _, r in qual[qual["pode_executar"] == 1].iterrows()
)

# Qual[(c,f)] = 1 se o colaborador c tem a qualificação necessária para a função f
#             = 0 caso contrário
# Corresponde ao parâmetro Qual_{c,f} da formulação matemática
Qual = {(c, f): 1 if (c, map_funcao_qual[f]) in qual_set else 0 for c in C for f in F}

# Cf[f] = lista de colaboradores qualificados para a função f
# Usado em H1: Σ_{c ∈ Cf[f]} x_{c,d,t,f} ≥ N_min_{d,t,f}
Cf = {f: [c for c in C if Qual[(c, f)] == 1] for f in F}

print("Colaboradores qualificados por função:")
for f, cf in Cf.items():
    print(f"  {f}: {len(cf)} colaboradores")

Colaboradores qualificados por função:
  Auxiliar de limpeza: 8 colaboradores
  Empregada de andares: 18 colaboradores
  Supervisora: 11 colaboradores


## 5b. Elegibilidade funcional hierárquica

O parâmetro `Qual(c,f)` construído na célula anterior regista apenas se o colaborador
tem a **qualificação técnica** para executar uma função — não considera a hierarquia
operacional. Uma Auxiliar de limpeza pode tecnicamente ter a qualificação registada,
mas não deve ser elegível para supervisão. Uma Supervisora pode substituir uma
Empregada de andares em situação de necessidade, mas não o inverso.

A P4 introduz `Elegivel(c,f)` que combina dois critérios:
1. **Qualificação técnica:** `Qual(c,f) = 1`
2. **Hierarquia funcional:** o perfil contratual do colaborador consta da lista de perfis
   elegíveis para a função (`funcao_principal` da tabela de colaboradores)

Apenas colaboradores que satisfaçam **ambos** os critérios são incluídos em `Cf[f]`.

### Matriz de elegibilidade funcional

| Função a executar | Perfis elegíveis | Lógica |
|---|---|---|
| **Supervisora** | Supervisora, Governanta | Só perfis de chefia/supervisão supervisionam |
| **Empregada de andares** | Empregada de andares, Supervisora, Governanta | Chefias podem executar operacionalmente se necessário |
| **Auxiliar de limpeza** | Auxiliar de limpeza, Empregada de andares, Supervisora, Governanta | Função mais abrangente — qualquer perfil superior pode executar |

### O que fica para evolução futura
- Penalização por desvio da função principal (Supervisora alocada a função operacional)
- Limite máximo de turnos de chefia em funções operacionais
- Regra de supervisão mínima por turno (pelo menos uma Supervisora efectivamente a supervisionar)

In [50]:
# ── Elegibilidade funcional hierárquica ──────────────────────────────────────

# Hierarquia funcional: por função a executar, que perfis contratuais são elegíveis
# Baseia-se na realidade operacional de Housekeeping — não na qualificação técnica isolada
ELEGIBILIDADE_HIERARQUICA = {
    "Supervisora"         : ["Supervisora", "Governanta"],
    "Empregada de andares": ["Empregada de andares", "Supervisora", "Governanta"],
    "Auxiliar de limpeza" : ["Auxiliar de limpeza", "Empregada de andares",
                             "Supervisora",          "Governanta"],
}

# Perfil contratual de cada colaborador (coluna funcao_principal da tabela colaboradores)
# É o perfil para o qual o colaborador foi contratado — determina a sua posição na hierarquia
Perfil_c = dict(zip(colab["colaborador_id"], colab["funcao_principal"]))

# Elegivel(c,f): 1 se o colaborador satisfaz AMBOS os critérios:
#   (a) qualificação técnica para f  →  Qual[(c,f)] = 1
#   (b) perfil contratual elegível para f  →  hierarquia respeitada
# Se qualquer um dos critérios falha → Elegivel = 0
Elegivel = {
    (c, f): 1
    if Qual.get((c, f), 0) == 1
    and Perfil_c.get(c, "") in ELEGIBILIDADE_HIERARQUICA.get(f, [])
    else 0
    for c in C for f in F
}

# Recalcular Cf com elegibilidade hierárquica
# Cf[f] passa a conter apenas colaboradores elegíveis (técnica + hierarquia)
# Todo o modelo usa Cf[f] — não é necessária mais nenhuma alteração
Cf = {f: [c for c in C if Elegivel.get((c, f), 0) == 1] for f in F}

# ── Verificação: comparar Cf antes e depois da P4 ────────────────────────────
print("Elegibilidade funcional hierárquica aplicada")
print()
print(f"  {'Função':<25} {'Qual (técnica)':>16} {'Elegivel (P4)':>15} {'Δ':>5}")
print("  " + "─" * 65)
for f in F:
    cf_qual     = [c for c in C if Qual.get((c, f), 0) == 1]
    cf_elegivel = Cf[f]
    delta       = len(cf_qual) - len(cf_elegivel)
    flag        = f"  ← -{delta}" if delta > 0 else "  ✓"
    print(f"  {f:<25} {len(cf_qual):>16} {len(cf_elegivel):>15}{flag}")

print()
print("  Colaboradores excluídos pela hierarquia (por função):")
for f in F:
    cf_qual     = [c for c in C if Qual.get((c, f), 0) == 1]
    cf_elegivel = set(Cf[f])
    excluidos   = [c for c in cf_qual if c not in cf_elegivel]
    if excluidos:
        for c in excluidos:
            print(f"    {f}: {c} (perfil={Perfil_c.get(c,'?')}) excluído por hierarquia")
    else:
        print(f"    {f}: sem exclusões")

print()
print("  Perfis por função elegível (após P4):")
for f in F:
    perfis = sorted(set(Perfil_c[c] for c in Cf[f]))
    print(f"    {f}: {perfis}")

Elegibilidade funcional hierárquica aplicada

  Função                      Qual (técnica)   Elegivel (P4)     Δ
  ─────────────────────────────────────────────────────────────────
  Auxiliar de limpeza                      8               8  ✓
  Empregada de andares                    18              17  ← -1
  Supervisora                             11              10  ← -1

  Colaboradores excluídos pela hierarquia (por função):
    Auxiliar de limpeza: sem exclusões
    Empregada de andares: C019 (perfil=Auxiliar de limpeza) excluído por hierarquia
    Supervisora: C006 (perfil=Empregada de andares) excluído por hierarquia

  Perfis por função elegível (após P4):
    Auxiliar de limpeza: ['Auxiliar de limpeza', 'Empregada de andares', 'Governanta', 'Supervisora']
    Empregada de andares: ['Empregada de andares', 'Governanta', 'Supervisora']
    Supervisora: ['Governanta', 'Supervisora']


## 6. Disponibilidade, cobertura e preferências

Nesta célula são construídos os restantes parâmetros do modelo — todos como dicionários indexados por tuplos.

**Porquê dicionários com tuplos como chaves?**
É a forma mais directa de implementar a notação matemática. Por exemplo, `Disp[(c,d,t)]` em Python corresponde exactamente a $Disp_{c,d,t}$ na fórmula matemática.

**FolgaMatch** merece atenção especial: converte o dia da semana preferido para folga (armazenado como número 0-6 na tabela) em um flag binário por (colaborador, data), indicando se aquela data coincide com o dia preferido.

In [51]:
# ── Disp: disponibilidade de cada colaborador por dia e turno ─────────────────
# Filtrar a tabela de disponibilidade para a semana em análise e os turnos T1/T2
disp_w = disp[
    (disp["data"].isin(datas)) &
    (disp["turno_id"].isin(T))
].copy()

# Disp[(c, d, t)] = 1 se o colaborador c está disponível no dia d, turno t
#                 = 0 se estiver indisponível (férias, baixa, etc.)
# H3 é garantida implicitamente: as variáveis x só existem onde Disp=1
Disp = {
    (r["colaborador_id"], r["data"].strftime("%Y-%m-%d"), r["turno_id"]): r["disponivel"]
    for _, r in disp_w.iterrows()
}

# ── N_min e N_ideal: necessidades de cobertura por dia, turno e função ─────────
cob_w = cob[
    (cob["data"].isin(datas)) &
    (cob["turno_id"].isin(T))
].copy()

# N_min[(d,t,f)]: mínimo de colaboradores exigidos — violação é proibida (hard)
N_min = {
    (r["data"].strftime("%Y-%m-%d"), r["turno_id"], r["funcao"]): r["colaboradores_minimos"]
    for _, r in cob_w.iterrows()
}

# N_ideal[(d,t,f)]: número ideal de colaboradores — afastamento é penalizado (soft)
N_ideal = {
    (r["data"].strftime("%Y-%m-%d"), r["turno_id"], r["funcao"]): r["colaboradores_ideais"]
    for _, r in cob_w.iterrows()
}

# ── Pref: score de preferência de turno por colaborador ───────────────────────
# Score 0 = evitar | 1 = indiferente | 2 = preferido
# A penalização S1 será (2 - Pref[c,t]) × x: quanto menor o score, maior a penalização
pref_dict = {(r["colaborador_id"], r["turno_id"]): r["score_preferencia_turno"]
             for _, r in pref.iterrows()}
Pref = {(c, t): pref_dict.get((c, t), 1) for c in C for t in T}
# Nota: default 1 (indiferente) para pares sem registo explícito

# ── FolgaMatch: flag que indica se a data coincide com o dia preferido de folga ─
# folga_preferida_dow é um número 0-6 (0=Segunda, 6=Domingo)
# dayofweek() do pandas retorna o mesmo formato
folga_pref = pref.groupby("colaborador_id")["folga_preferida_dow"].first().to_dict()
FolgaMatch = {
    (c, d): 1 if pd.Timestamp(d).dayofweek == folga_pref.get(c, -1) else 0
    for c in C for d in D
}

# ── Verificação de consistência ───────────────────────────────────────────────
disponiveis = sum(1 for v in Disp.values() if v == 1)
print(f"Disp total: {len(Disp)} | Disponíveis (=1): {disponiveis}")
print(f"N_min: {len(N_min)} | N_ideal: {len(N_ideal)}")
print(f"Pref: {len(Pref)} | FolgaMatch: {len(FolgaMatch)}")

# Aviso automático se Disp estiver vazio (problema de datas)
if len(Disp) == 0:
    print("\n⚠️  ATENÇÃO: Disp está vazio!")
    print(f"  Datas no ficheiro: {disp['data'].min().date()} a {disp['data'].max().date()}")
    print(f"  Datas pedidas: {D[0]} a {D[-1]}")

Disp total: 280 | Disponíveis (=1): 264
N_min: 42 | N_ideal: 42
Pref: 40 | FolgaMatch: 140


## 6b. Integração Bloco 3 → Bloco 4

O modelo treinado e guardado no Bloco 3 é carregado aqui directamente — sem re-treino.
A semana-alvo é lida automaticamente do ficheiro `gb_config.pkl`, definido no início do Bloco 3.

**Para mudar a semana:** alterar `SEMANA_PREV_INICIO` e `SEMANA_PREV_FIM` no Bloco 3,
correr as células de treino e guardar, e depois correr o Bloco 4 — as datas propagam
automaticamente.

Os valores de `N_min` e `N_ideal` são substituídos pelas previsões do modelo,
tornando o optimizer independente dos dados históricos de cobertura.

In [52]:
import numpy as np

# ── 1. Carregar modelos e features guardados no Bloco 3 ──────────────────────
# datas, D e SEMANA_INICIO já estão definidos na célula 3 (Conjuntos)
gb_model    = joblib.load(DATA_DIR_OUTPUTS / "gb_model.pkl")     # v1: prevê colaboradores_minimos → N_min
gb_model_v2 = joblib.load(DATA_DIR_OUTPUTS / "gb_model_v2.pkl")  # prevê colaboradores_ideais → N_ideal
gb_features = joblib.load(DATA_DIR_OUTPUTS / "gb_features.pkl")

print(f"Modelo v1 (N_min)   : {type(gb_model).__name__}")
print(f"Modelo ideal (N_ideal): {type(gb_model_v2).__name__}")
print(f"Features          : {len(gb_features)} colunas")
print(f"Semana-alvo       : {SEMANA_INICIO} a {SEMANA_FIM}")

# ── 2. Construir features operacionais da semana-alvo ────────────────────────
ativ_hotel  = pd.read_csv(DATA_DIR / "atividade_hotel.csv",     parse_dates=["data"])
atv_tip     = pd.read_csv(DATA_DIR / "atividade_tipologia.csv", parse_dates=["data"])
cob_full    = pd.read_csv(DATA_DIR / "cobertura_necessidades.csv", parse_dates=["data"])
cob_full    = cob_full[cob_full["turno_id"].isin(T)].copy()

atv_tip_agg = atv_tip.groupby("data", as_index=False).agg(
    carga_total_min        = ("carga_limpeza_min",   "sum"),
    quartos_checkout_total = ("quartos_checkout",    "sum"),
    quartos_stayover_total = ("quartos_stayover",    "sum")
)
ativ_semana = ativ_hotel[ativ_hotel["data"].isin(datas)].merge(
    atv_tip_agg, on="data", how="left"
)
cob_semana = cob_full[cob_full["data"].isin(datas)].copy()
df_alvo    = cob_semana.merge(ativ_semana, on="data", how="left")
if "dia_semana_x" in df_alvo.columns:
    df_alvo = df_alvo.rename(columns={"dia_semana_y": "dia_semana"}).drop(columns=["dia_semana_x"])

# ── 3. Encoding consistente com o treino do Bloco 3 ──────────────────────────
features_gb = ["turno_id","funcao","taxa_ocupacao","quartos_ocupados","n_checkins",
    "n_checkouts","evento_especial","tipo_dia","epoca","mes","dia_semana",
    "quartos_a_limpar_estimados","carga_total_min","quartos_checkout_total","quartos_stayover_total"]
cat_cols = ["turno_id","funcao","tipo_dia","epoca","dia_semana"]

X_alvo = pd.get_dummies(df_alvo[features_gb], columns=cat_cols, drop_first=False)
# reindex: garante mesmas colunas e ordem que o treino
# colunas ausentes (categorias não presentes nesta semana) ficam a zero
X_alvo = X_alvo.reindex(columns=gb_features, fill_value=0)

# ── 4. Prever N_min e substituir nos parâmetros do optimizer ─────────────────
y_pred     = gb_model.predict(X_alvo)
y_pred_int = np.maximum(1, np.round(y_pred)).astype(int)  # round: N_min mais próximo do real

df_alvo["N_min_previsto"] = y_pred_int
df_alvo["N_min_real"]     = df_alvo["colaboradores_minimos"].values
df_alvo["diff"]           = df_alvo["N_min_previsto"] - df_alvo["N_min_real"]

N_min = {
    (r["data"].strftime("%Y-%m-%d"), r["turno_id"], r["funcao"]): int(r["N_min_previsto"])
    for _, r in df_alvo.iterrows()
}
# ── N_ideal previsto pelo segundo modelo ─────────────────────────────────────
y_pred_ideal = gb_model_v2.predict(X_alvo)
N_ideal = {
    (r["data"].strftime("%Y-%m-%d"), r["turno_id"], r["funcao"]): int(np.maximum(1, np.ceil(p)))
    for (_, r), p in zip(df_alvo.iterrows(), y_pred_ideal)
}

# ── 5. Resumo ─────────────────────────────────────────────────────────────────
iguais = (df_alvo["diff"] == 0).sum()
print(f"\nSlots previstos    : {len(N_min)}")
print(f"Previsões exactas  : {iguais}/{len(N_min)} ({iguais/len(N_min)*100:.0f}%)")
print(f"N_min total        : {sum(N_min.values())} turnos")
print(f"Capacidade interna : {sum(W_c[c] for c in C)} turnos/semana")
if sum(N_min.values()) > sum(W_c[c] for c in C):
    print("\n⚠️  N_min excede capacidade — optimizer pode ser Infeasible.")
    print("   Considerar a extensão com recursos externos (secções 15-18).")
print("\nN_min e N_ideal actualizados com previsões GB — pronto para o optimizer.")


Modelo v1 (N_min)   : GradientBoostingRegressor
Modelo ideal (N_ideal): GradientBoostingRegressor
Features          : 29 colunas
Semana-alvo       : 2025-01-01 a 2025-01-07

Slots previstos    : 42
Previsões exactas  : 38/42 (90%)
N_min total        : 77 turnos
Capacidade interna : 96 turnos/semana

N_min e N_ideal actualizados com previsões GB — pronto para o optimizer.


> **Nota de consistência:** as variáveis de decisão internas `x_{c,d,t,f}` são
> criadas com `Elegivel(c,f) == 1` em vez de `Qual(c,f) == 1`. Isto garante que a
> elegibilidade funcional hierárquica da P4 restringe efectivamente o domínio das
> variáveis — e não apenas os conjuntos `Cf[f]`. A P4 só fica completamente
> implementada quando ambos são consistentes: `Cf[f]` e as variáveis `x`.

## 7. Modelo e variáveis de decisão

### O problema ILP

Um problema ILP (Integer Linear Programming) tem três componentes:
1. **Variáveis de decisão** — o que o solver pode escolher
2. **Restrições** — limites que a solução tem de respeitar
3. **Função objetivo** — o que queremos minimizar

### A variável de decisão

$$x_{c,d,t,f} \in \{0, 1\}$$

- **= 1** se o colaborador *c* é alocado ao dia *d*, turno *t*, função *f*
- **= 0** caso contrário

Sem filtros, haveria $|C| \times |D| \times |T| \times |F|$ variáveis. Com os filtros de disponibilidade e qualificação, o número reduz significativamente — o que acelera o solver.

### Os pesos da função objetivo

| Peso | Valor | Controla |
|---|---|---|
| w1 | 5 | Penalização de preferências (S1) e folgas (S2) |
| w2 | 10 | Penalização de défice face ao ideal (S4) |
| w3 | 2 | Penalização de desequilíbrio de carga (S3) |

Valores maiores = maior prioridade. w2 > w1 > w3 significa que atingir a cobertura ideal é mais importante do que respeitar preferências, que por sua vez é mais importante do que a equidade.

In [53]:
# ── Pesos da função objetivo (ajustáveis) ─────────────────────────────────────
w1 = 5    # penalização preferências de turno (S1) e folga preferida (S2)
w2 = 10   # penalização défice cobertura ideal (S4)
w3 = 2    # penalização desequilíbrio de carga (S3)

# ── Custo fixo semanal da equipa interna (comprometido, fora da função objetivo) ─
# Calculado aqui para ser reportado nos KPIs
# Não entra na função objetivo — é custo já comprometido
H_t_medio_v1     = (H_t["T1"] + H_t["T2"]) / 2
Custo_fixo_v1    = {c: Cost_c[c] * H_t_medio_v1 * W_c[c] for c in C}
Custo_fixo_total_v1 = sum(Custo_fixo_v1.values())

# ── Criar o problema ILP ───────────────────────────────────────────────────────
prob = LpProblem("HousekeepingSchedule_base", LpMinimize)

# ── Variáveis de decisão x_{c,d,t,f} ────────────────────────────────────────
x = {
    (c, d, t, f): LpVariable(f"x_{c}_{d}_{t}_{f}", cat="Binary")
    for c in C for d in D for t in T for f in F
    if Disp.get((c, d, t), 0) == 1 and Elegivel.get((c, f), 0) == 1
}

print(f"Variáveis de decisão: {len(x)}")
print(f"(Sem filtro seriam  : {len(C)*len(D)*len(T)*len(F)})")
print(f"Redução             : {(1 - len(x)/(len(C)*len(D)*len(T)*len(F)))*100:.1f}%")
print(f"Custo fixo total equipa: {Custo_fixo_total_v1:.2f}€ (comprometido, fora da obj.)")

if len(x) == 0:
    print("\n⚠️  ATENÇÃO: nenhuma variável criada — verificar Disp e Qual acima")

Variáveis de decisão: 466
(Sem filtro seriam  : 840)
Redução             : 44.5%
Custo fixo total equipa: 6428.02€ (comprometido, fora da obj.)


## 8. Hard constraints

As hard constraints são restrições **obrigatórias** — qualquer solução que as viole é inválida. O solver CBC nunca retorna uma solução que viole uma hard constraint.

| Constraint | Fórmula resumida | Significado |
|---|---|---|
| **H1** | Σ x ≥ N_min | Cobertura mínima garantida em cada slot |
| **H2** | Σ x ≤ 1 por (c,d) | Cada colaborador trabalha no máximo 1 turno/dia |
| **H5** | Σ H_t × x ≤ H_c | Horas do turno não excedem o máximo contratual diário |
| **H6** | Σ x ≤ W_c por semana | Dias trabalhados não excedem o máximo semanal |
| **H10** | Σ x ≤ M_c por janela | Não mais de M_c dias consecutivos trabalhados |

**Nota:** H3 (disponibilidade) e H8 (qualificação) não aparecem aqui porque foram garantidas implicitamente na criação das variáveis — só existem variáveis x para combinações válidas.

In [54]:
# ── H1: Cobertura mínima por dia, turno e função ──────────────────────────────
# Para cada combinação (d,t,f), a soma dos colaboradores qualificados alocados
# deve ser ≥ ao mínimo definido na tabela de cobertura
# Fórmula: Σ_{c ∈ Cf[f]} x_{c,d,t,f} ≥ N_min_{d,t,f}
for d in D:
    for t in T:
        for f in F:
            # al = lista de variáveis x disponíveis para esta combinação (d,t,f)
            al = [x[(c,d,t,f)] for c in Cf[f] if (c,d,t,f) in x]
            if al and (d,t,f) in N_min:
                prob += lpSum(al) >= N_min[(d,t,f)], f"H1_{d}_{t}_{f}"

# ── H2: Máximo de um turno por colaborador por dia ────────────────────────────
# Garante que nenhum colaborador é alocado a dois turnos no mesmo dia
# Fórmula: Σ_{t,f} x_{c,d,t,f} ≤ 1  ∀ c,d
for c in C:
    for d in D:
        cv = [x[(c,d,t,f)] for t in T for f in F if (c,d,t,f) in x]
        if cv:
            prob += lpSum(cv) <= 1, f"H2_{c}_{d}"

# ── H5: Limite de horas diárias por contrato ──────────────────────────────────
# A duração do turno alocado não pode exceder o máximo diário do colaborador
# Fórmula: Σ_{t,f} H_t × x_{c,d,t,f} ≤ H_c  ∀ c,d
# Nota: como H2 já garante ≤1 turno/dia, esta restrição verifica apenas
# se a duração do turno escolhido é compatível com o contrato
for c in C:
    for d in D:
        cv = [H_t[t] * x[(c,d,t,f)] for t in T for f in F if (c,d,t,f) in x]
        if cv:
            prob += lpSum(cv) <= H_c[c], f"H5_{c}_{d}"

# ── H6: Limite de dias de trabalho por semana ─────────────────────────────────
# O número total de dias trabalhados na semana não pode exceder W_c
# Fórmula: Σ_{d,t,f} x_{c,d,t,f} ≤ W_c  ∀ c
# (como H2 garante ≤1 turno/dia, esta soma é igual ao nº de dias trabalhados)
for c in C:
    cv = [x[(c,d,t,f)] for d in D for t in T for f in F if (c,d,t,f) in x]
    if cv:
        prob += lpSum(cv) <= W_c[c], f"H6_{c}"

# ── H10: Máximo de dias consecutivos ─────────────────────────────────────────
# Para qualquer janela de M_c+1 dias consecutivos, no máximo M_c podem ser trabalhados
# Fórmula: Σ_{d'=d}^{d+M_c} Σ_{t,f} x_{c,d',t,f} ≤ M_c  ∀ c, d
# Exemplo: se M_c=5, uma janela de 6 dias pode ter no máximo 5 dias de trabalho
for c in C:
    mc = int(M_c[c])   # max dias consecutivos deste colaborador
    for start in range(len(D) - mc):
        window = D[start : start + mc + 1]   # janela de mc+1 dias
        cv = [x[(c,d,t,f)] for d in window for t in T for f in F if (c,d,t,f) in x]
        if cv:
            prob += lpSum(cv) <= mc, f"H10_{c}_{start}"

print(f"Restrições adicionadas ao modelo: {len(prob.constraints)}")

Restrições adicionadas ao modelo: 401


## 9. Variáveis auxiliares e soft constraints

Ao contrário das hard constraints, as soft constraints **podem ser violadas** — mas cada violação é penalizada na função objetivo. O solver tenta minimizar essas penalizações, mas pode aceitá-las se o benefício compensar.

### Como se implementam soft constraints num ILP?

São transformadas em **variáveis auxiliares contínuas** (≥ 0) que capturam o grau de violação:

**S4 — Défice face ao ideal:**
A variável $\delta_{d,t,f} \geq 0$ mede quantos colaboradores faltam para atingir o ideal.
A restrição garante que $\delta$ é pelo menos igual ao défice real: $\delta \geq N_{ideal} - \sum x$

**S3 — Equidade de carga:**
A variável $e_c \geq 0$ mede o desvio de cada colaborador face à média $\mu$.
Duas restrições garantem que $e_c$ captura o desvio em ambas as direcções: $e_c \geq \text{turnos}_c - \mu$ e $e_c \geq \mu - \text{turnos}_c$

**S1 e S2** não precisam de variáveis auxiliares — são calculadas directamente na função objetivo.

In [55]:
# ── μ: média teórica de turnos por colaborador na semana ──────────────────────
# Calculada como: (dias na semana × capacidade semanal média) / 7
# Serve de referência para a restrição de equidade (S3)
mu = len(D) * sum(W_c[c] for c in C) / len(C) / 7

# ── δ: variáveis de défice para S4 (cobertura ideal) ─────────────────────────
# δ_{d,t,f} ≥ 0 representa o défice face ao ideal em cada slot (d,t,f)
# Se alocados ≥ ideal → δ = 0 (sem penalização)
# Se alocados < ideal → δ > 0 (penalizado com peso w2 na função objetivo)
delta = {
    (d, t, f): LpVariable(f"delta_{d}_{t}_{f}", lowBound=0)
    for d in D for t in T for f in F
    if (d, t, f) in N_ideal
}

# Restrição S4: δ_{d,t,f} ≥ N_ideal_{d,t,f} - Σ x_{c,d,t,f}
# Implementação robusta: a restrição é sempre criada quando (d,t,f) ∈ N_ideal ∩ delta,
# mesmo que al == [] (nenhum colaborador qualificado disponível nesse slot).
# Nesse caso lpSum([]) = 0 e a restrição torna-se δ ≥ N_ideal — penaliza totalmente.
for d in D:
    for t in T:
        for f in F:
            if (d, t, f) in N_ideal and (d, t, f) in delta:
                al = [x[(c,d,t,f)] for c in Cf[f] if (c,d,t,f) in x]
                # lpSum(al) funciona correctamente mesmo se al == []
                prob += delta[(d,t,f)] >= N_ideal[(d,t,f)] - lpSum(al), f"S4_{d}_{t}_{f}"

# ── e: variáveis de desvio para S3 (equidade de carga) ───────────────────────
# e_c ≥ 0 representa o desvio do colaborador c face à média μ
# Lineariza o valor absoluto: e_c ≥ |turnos_c - μ|
e = {c: LpVariable(f"e_{c}", lowBound=0) for c in C}

for c in C:
    tc = [x[(c,d,t,f)] for d in D for t in T for f in F if (c,d,t,f) in x]
    if tc:
        prob += e[c] >= lpSum(tc) - mu, f"S3a_{c}"
        prob += e[c] >= mu - lpSum(tc), f"S3b_{c}"

print(f"μ (turnos médios/colaborador esperados na semana): {mu:.2f}")
print(f"Variáveis auxiliares: {len(delta)} delta + {len(e)} e_c")
print(f"S4 robusto: restrição criada para todos os {len(delta)} slots (mesmo com al vazio)")
print(f"Total de restrições no modelo: {len(prob.constraints)}")

μ (turnos médios/colaborador esperados na semana): 4.80
Variáveis auxiliares: 42 delta + 20 e_c
S4 robusto: restrição criada para todos os 42 slots (mesmo com al vazio)
Total de restrições no modelo: 483


## 10. Função objetivo

### Formulação

A função objetivo combina cinco componentes. O custo interno **não entra como custo financeiro real** — é tratado como um *proxy de utilização* que orienta a preferência do solver entre colaboradores equivalentes, sem distorcer a comparação económica entre cenários.

$$\min Z = \underbrace{\sum_{c,d,t,f} Cost_c \cdot H_t \cdot x_{c,d,t,f}}_{\text{proxy de utilização}}
+ w_1 \cdot \underbrace{\sum_{c,d,t,f}(2 - Pref_{c,t}) \cdot x_{c,d,t,f}}_{\text{S1: preferências de turno}}
+ w_1 \cdot \underbrace{\sum_{c,d,t,f} FolgaMatch_{c,d} \cdot x_{c,d,t,f}}_{\text{S2: folga preferida}}
+ w_2 \cdot \underbrace{\sum_{d,t,f} \delta_{d,t,f}}_{\text{S4: défice face ao ideal}}
+ w_3 \cdot \underbrace{\sum_c e_c}_{\text{S3: equidade de carga}}$$

### Papel de cada componente

| Componente | Natureza | Controla |
|---|---|---|
| Proxy de utilização | Score de ordenação entre colaboradores | Prefere colaboradores com menor custo/hora quando equivalentes |
| $w_1 \cdot (\text{S1}+\text{S2})$ | Penalização de qualidade | Viola preferências de turno e dias de folga |
| $w_2 \cdot \text{S4}$ | Penalização de cobertura | Afastamento face ao número ideal de colaboradores |
| $w_3 \cdot \text{S3}$ | Penalização de equidade | Desequilíbrio na distribuição de carga entre colaboradores |

### Custo fixo interno — fora da função objetivo

O custo salarial dos colaboradores internos é tratado como **fixo e já comprometido**. Não entra na função objetivo porque não depende das decisões de alocação — é igual independentemente de quantos turnos cada colaborador trabalha na semana. É reportado separadamente nos KPIs como `Custo fixo comprometido`.

### Pesos da função objetivo

Os pesos `w1`, `w2`, `w3` são calibrados na secção 19 (análise de sensibilidade). Os valores actuais resultam dessa calibração.

In [56]:
# ── Função objetivo: custo externo incremental + penalizações ────────────────
# O custo interno é fixo e já comprometido — não deve guiar decisões de alocação.
# É mantido como proxy de utilização: preferir colaboradores com menor custo/hora
# quando tudo o resto é equivalente. Este efeito é secundário face às penalizações.

# Proxy de utilização interna (ordena preferência entre colaboradores equivalentes)
proxy_utilizacao = lpSum(Cost_c[c] * H_t[t] * x[(c,d,t,f)] for (c,d,t,f) in x)

# Penalização de preferências de turno — S1
pen_preferencias = lpSum((2 - Pref[(c,t)]) * x[(c,d,t,f)] for (c,d,t,f) in x)

# Penalização de folga no dia preferido — S2
pen_folga = lpSum(FolgaMatch[(c,d)] * x[(c,d,t,f)] for (c,d,t,f) in x)

# Défice face à cobertura ideal — S4
pen_deficit = lpSum(delta[(d,t,f)] for (d,t,f) in delta)

# Desequilíbrio de carga — S3
pen_equidade = lpSum(e[c] for c in C)

# ── Função objetivo: proxy + penalizações (sem custo financeiro fixo) ─────────
# Nota: proxy_utilizacao tem escala em euros mas é tratado como score de preferência,
# não como custo real. O custo real (fixo) é reportado separadamente nos KPIs.
prob += (
    proxy_utilizacao
    + w1 * (pen_preferencias + pen_folga)
    + w2 * pen_deficit
    + w3 * pen_equidade
)

print(f"Função objetivo definida.")
print(f"  Proxy utilização : incluído (score de preferência, não custo financeiro)")
print(f"  Pesos: w1={w1} (preferências/folga) | w2={w2} (défice ideal) | w3={w3} (equidade)")
print(f"  Total variáveis  : {len(prob.variables())}")
print(f"  Total restrições : {len(prob.constraints)}")

Função objetivo definida.
  Proxy utilização : incluído (score de preferência, não custo financeiro)
  Pesos: w1=5 (preferências/folga) | w2=10 (défice ideal) | w3=2 (equidade)
  Total variáveis  : 528
  Total restrições : 483


## 11. Resolver o modelo

O solver CBC (COIN Branch and Cut) é chamado para encontrar a solução óptima. Para problemas ILP pequenos como este (< 1000 variáveis, < 300 restrições), o CBC resolve tipicamente em menos de 1 segundo.

**Interpretação do status:**
- `Optimal` — solução óptima encontrada (o melhor resultado possível)
- `Infeasible` — não existe solução que satisfaça todas as hard constraints
- `Unbounded` — problema mal definido (não deve acontecer com este modelo)

**Interpretação de Z:**
O valor de Z é a soma ponderada de custo + penalizações. Não é directamente interpretável em euros — serve apenas para comparar soluções entre si.

In [57]:
# Resolver o modelo com o solver CBC
# msg=0 suprime o log detalhado do solver (mudar para msg=1 para ver o processo)
t0 = time.time()
prob.solve(PULP_CBC_CMD(msg=0))
elapsed = time.time() - t0

print(f"Status : {LpStatus[prob.status]}")
print(f"Tempo  : {elapsed:.2f}s")
print(f"Z      : {value(prob.objective):.2f}  (custo + penalizações ponderadas)")

if LpStatus[prob.status] != "Optimal":
    print("\n⚠️  Solução não óptima. Verificar disponibilidades e restrições.")

Status : Optimal
Tempo  : 0.29s
Z      : 5307.48  (custo + penalizações ponderadas)


## 12. Extrair a escala gerada

Após a resolução, percorremos todas as variáveis de decisão e extraímos as que o solver fixou em 1 (alocadas). O resultado é um DataFrame com uma linha por alocação.

In [58]:
# Extrair todas as alocações onde x=1
# value(var) retorna o valor da variável após resolução (0 ou 1 para binárias)
# Usamos > 0.5 em vez de == 1 para evitar problemas de arredondamento numérico
rows = [
    {"colaborador_id": c, "data": d, "turno_id": t, "funcao": f}
    for (c,d,t,f), var in x.items()
    if value(var) is not None and value(var) > 0.5
]

escala = pd.DataFrame(rows)
print(f"Alocações geradas: {len(escala)}")
print(f"(Cada linha = 1 colaborador alocado a 1 dia + turno + função)")
escala.head(10)

Alocações geradas: 77
(Cada linha = 1 colaborador alocado a 1 dia + turno + função)


,colaborador_id,data,turno_id,funcao
0,C002,2025-01-03,T2,Empregada de andares
1,C002,2025-01-04,T2,Supervisora
2,C002,2025-01-05,T2,Empregada de andares
3,C002,2025-01-06,T2,Empregada de andares
4,C002,2025-01-07,T2,Supervisora
5,C003,2025-01-02,T1,Empregada de andares
6,C003,2025-01-04,T1,Empregada de andares
7,C003,2025-01-05,T1,Empregada de andares
8,C003,2025-01-06,T1,Empregada de andares
9,C003,2025-01-07,T1,Empregada de andares


In [59]:
# ── Pivot para formato semanal legível ────────────────────────────────────────
# Transforma o DataFrame em formato de tabela: colaboradores nas linhas, dias/turnos nas colunas
if escala.empty:
    print("⚠️  Escala vazia — verificar resolução.")
else:
    # Abreviar os nomes das funções para caber na tabela
    escala["abrev"] = escala["funcao"].map({
        "Empregada de andares": "Emp",   # Empregada de andares
        "Supervisora"         : "Sup",   # Supervisora
        "Auxiliar de limpeza" : "Aux"    # Auxiliar de limpeza
    })

    # Criar tabela pivô: linhas = colaborador, colunas = (data, turno)
    pivot = escala.pivot_table(
        index="colaborador_id",
        columns=["data","turno_id"],
        values="abrev",
        aggfunc=lambda x: x.iloc[0]   # em cada célula: a função alocada
    )

    print("Escala semanal (Emp=Empregada de andares, Sup=Supervisora, Aux=Auxiliar, - =folga):")
    print(pivot.fillna("-").to_string())   # preencher células vazias com "-" (folga)

Escala semanal (Emp=Empregada de andares, Sup=Supervisora, Aux=Auxiliar, - =folga):
data           2025-01-01      2025-01-02      2025-01-03      2025-01-04      2025-01-05      2025-01-06      2025-01-07     
turno_id               T1   T2         T1   T2         T1   T2         T1   T2         T1   T2         T1   T2         T1   T2
colaborador_id                                                                                                                
C002                    -    -          -    -          -  Emp          -  Sup          -  Emp          -  Emp          -  Sup
C003                    -    -        Emp    -          -    -        Emp    -        Emp    -        Emp    -        Emp    -
C004                    -    -        Emp    -        Emp    -          -    -        Emp    -        Emp    -        Emp    -
C005                  Aux    -        Aux    -        Aux    -        Aux    -          -    -        Aux    -          -    -
C006                    -  

## 13. KPIs da solução

Os KPIs permitem avaliar a qualidade da escala gerada de forma operacionalmente interpretável — ao contrário do valor de Z (que é abstracto), os KPIs têm significado directo para um gestor de Housekeeping.

In [60]:
if not escala.empty:

    # ── Custo total da escala (€) ─────────────────────────────────────────────
    # Soma de: custo/hora × horas do turno, para cada alocação
    custo_total = sum(
        Cost_c[r["colaborador_id"]] * H_t[r["turno_id"]]
        for _, r in escala.iterrows()
    )

    # ── Taxa de cobertura mínima ──────────────────────────────────────────────
    # Para cada slot (dia, turno, função): conta se o nº de alocados ≥ N_min
    slots_ok = sum(
        1 for (d,t,f), n in N_min.items()
        if len(escala[(escala["data"]==d) & (escala["turno_id"]==t) & (escala["funcao"]==f)]) >= n
    )

    # ── Défice total face ao ideal ────────────────────────────────────────────
    # Soma de max(0, N_ideal - alocados) em todos os slots
    # Se = 0: a escala atingiu o ideal em todos os slots
    deficit = sum(
        max(0, N_ideal.get((d,t,f), 0) -
            len(escala[(escala["data"]==d) & (escala["turno_id"]==t) & (escala["funcao"]==f)]))
        for d in D for t in T for f in F if (d,t,f) in N_ideal
    )

    # ── Distribuição de turnos por colaborador ────────────────────────────────
    tpc = escala.groupby("colaborador_id").size()

    # ── Apresentação dos KPIs ─────────────────────────────────────────────────
    print("=" * 45)
    print("  KPIs — Escala semanal (modelo base)")
    print("=" * 45)
    print(f"  Custo total            : {custo_total:.2f} €")
    print(f"  Cobertura mínima       : {slots_ok}/{len(N_min)} slots = {slots_ok/len(N_min)*100:.1f}%")
    print(f"  Violações H1           : {len(N_min)-slots_ok}  (0 = perfeito)")
    print(f"  Défice ideal (S4)      : {deficit}  (0 = ideal atingido em todos os slots)")
    print(f"  Total alocações        : {len(escala)}")
    print(f"  Colaboradores escalados: {escala['colaborador_id'].nunique()}/{len(C)}")
    print(f"  Turnos/colaborador     : min={tpc.min()} | max={tpc.max()} | média={tpc.mean():.1f}")
    print("=" * 45)

  KPIs — Escala semanal (modelo base)
  Custo total            : 4792.48 €
  Cobertura mínima       : 42/42 slots = 100.0%
  Violações H1           : 0  (0 = perfeito)
  Défice ideal (S4)      : 34  (0 = ideal atingido em todos os slots)
  Total alocações        : 77
  Colaboradores escalados: 16/20
  Turnos/colaborador     : min=2 | max=5 | média=4.8


---

# 14. Análise de sensibilidade — Pesos da função objetivo

## 14. Análise de sensibilidade aos pesos w1, w2, w3

A função objetivo combina quatro componentes ponderadas:

$$\min Z = \underbrace{\text{proxy}}_{\text{utilização}} + w_1 \cdot (\text{S1}+\text{S2}) + w_2 \cdot \text{S4} + w_3 \cdot \text{S3}$$

Os pesos actuais (`w1=5, w2=10, w3=2`) foram definidos como ponto de partida e são aqui calibrados empiricamente. A análise de sensibilidade responde à questão:

> **Que impacto têm diferentes valores de w1, w2 e w3 nos KPIs operacionais da escala?**

### Metodologia
- Semana de referência: definida no Bloco 3 (`gb_config.pkl`)
- 8 configurações de pesos testadas sistematicamente
- KPIs avaliados por configuração: proxy de custo, cobertura, défice ideal, equidade, violações de preferência
- A configuração de referência (actual) serve de baseline para comparação

### O que cada peso controla

| Peso | Soft constraint | Trade-off principal |
|---|---|---|
| w1 | S1 (preferências turno) + S2 (folga preferida) | Satisfação dos colaboradores vs. custo proxy |
| w2 | S4 (défice cobertura ideal) | Qualidade de cobertura vs. custo proxy |
| w3 | S3 (equidade de carga) | Distribuição justa vs. custo proxy |

In [61]:
# ── Análise de sensibilidade aos pesos w1, w2, w3 ──────────────────────────────
from pulp import LpProblem, LpMinimize, LpVariable, lpSum, LpStatus, value, PULP_CBC_CMD
import time

# Configurações de pesos a testar
# Cada tuplo: (w1, w2, w3, label, descrição)
configs_sensibilidade = [
    (5,  10, 2, "Referência",       "Configuração actual — baseline de comparação"),
    (1,  10, 2, "w1↓ preferências", "Menor peso nas preferências — mais custo-eficiente"),
    (10, 10, 2, "w1↑ preferências", "Maior peso nas preferências — mais satisfação"),
    (5,  5,  2, "w2↓ ideal",        "Menor peso no défice ideal — aceita mais afastamento"),
    (5,  20, 2, "w2↑ ideal",        "Maior peso no défice ideal — aproxima-se mais do ideal"),
    (5,  10, 1, "w3↓ equidade",     "Menor peso na equidade — permite maior desequilíbrio"),
    (5,  10, 8, "w3↑ equidade",     "Maior peso na equidade — distribui carga mais uniformemente"),
    (2,  15, 5, "Equilibrado+",     "Alternativa calibrada — reforça ideal e equidade"),
]

def resolver_com_pesos(w1_s, w2_s, w3_s):
    """Resolve o modelo v1 com pesos alternativos e devolve KPIs."""
    p = LpProblem(f"sens_{w1_s}_{w2_s}_{w3_s}", LpMinimize)

    # Variáveis (iguais ao modelo principal)
    x_s = {
        (c,d,t,f): LpVariable(f"xs_{c}_{d}_{t}_{f}", cat="Binary")
        for c in C for d in D for t in T for f in F
        if Disp.get((c,d,t),0)==1 and Elegivel.get((c,f),0)==1
    }
    delta_s = {(d,t,f): LpVariable(f"ds_{d}_{t}_{f}", lowBound=0)
               for d in D for t in T for f in F if (d,t,f) in N_ideal}
    e_s = {c: LpVariable(f"es_{c}", lowBound=0) for c in C}

    # Hard constraints (iguais ao modelo principal)
    for d in D:
        for t in T:
            for f in F:
                al = [x_s[(c,d,t,f)] for c in Cf[f] if (c,d,t,f) in x_s]
                if al and (d,t,f) in N_min:
                    p += lpSum(al) >= N_min[(d,t,f)]
    for c in C:
        for d in D:
            cv = [x_s[(c,d,t,f)] for t in T for f in F if (c,d,t,f) in x_s]
            if cv: p += lpSum(cv) <= 1
            cv2= [H_t[t]*x_s[(c,d,t,f)] for t in T for f in F if (c,d,t,f) in x_s]
            if cv2: p += lpSum(cv2) <= H_c[c]
        cv = [x_s[(c,d,t,f)] for d in D for t in T for f in F if (c,d,t,f) in x_s]
        if cv: p += lpSum(cv) <= W_c[c]
        mc = int(M_c[c])
        for start in range(len(D)-mc):
            window = D[start:start+mc+1]
            cv = [x_s[(c,d,t,f)] for d in window for t in T for f in F if (c,d,t,f) in x_s]
            if cv: p += lpSum(cv) <= mc
        tc = [x_s[(c,d,t,f)] for d in D for t in T for f in F if (c,d,t,f) in x_s]
        if tc:
            p += e_s[c] >= lpSum(tc) - mu, f"S3a_{c}"
            p += e_s[c] >= mu - lpSum(tc), f"S3b_{c}"
    for d in D:
        for t in T:
            for f in F:
                if (d,t,f) in N_ideal and (d,t,f) in delta_s:
                    al = [x_s[(c,d,t,f)] for c in Cf[f] if (c,d,t,f) in x_s]
                    if al: p += delta_s[(d,t,f)] >= N_ideal[(d,t,f)] - lpSum(al)

    # Função objetivo com pesos alternativos
    proxy  = lpSum(Cost_c[c]*H_t[t]*x_s[(c,d,t,f)] for (c,d,t,f) in x_s)
    pen_p  = lpSum((2-Pref[(c,t)])*x_s[(c,d,t,f)] for (c,d,t,f) in x_s)
    pen_f  = lpSum(FolgaMatch[(c,d)]*x_s[(c,d,t,f)] for (c,d,t,f) in x_s)
    pen_d  = lpSum(delta_s[(d,t,f)] for (d,t,f) in delta_s)
    pen_e  = lpSum(e_s[c] for c in C)
    p += proxy + w1_s*(pen_p+pen_f) + w2_s*pen_d + w3_s*pen_e

    p.solve(PULP_CBC_CMD(msg=0))
    if LpStatus[p.status] != "Optimal":
        return None

    rows_s = [{"colaborador_id":c,"data":d,"turno_id":t,"funcao":f}
              for (c,d,t,f),v in x_s.items() if value(v) is not None and value(v)>0.5]
    esc_s = pd.DataFrame(rows_s)
    if esc_s.empty: return None

    proxy_val = sum(Cost_c[r["colaborador_id"]]*H_t[r["turno_id"]] for _,r in esc_s.iterrows())
    ok_s      = sum(1 for (d,t,f),n in N_min.items()
                    if len(esc_s[(esc_s["data"]==d)&(esc_s["turno_id"]==t)&(esc_s["funcao"]==f)])>=n)
    deficit_s = sum(max(0, N_ideal.get((d,t,f),0)-
                        len(esc_s[(esc_s["data"]==d)&(esc_s["turno_id"]==t)&(esc_s["funcao"]==f)]))
                    for d in D for t in T for f in F if (d,t,f) in N_ideal)
    tpc_s     = esc_s.groupby("colaborador_id").size()
    pref_v    = sum((2-Pref[(r["colaborador_id"],r["turno_id"])]) for _,r in esc_s.iterrows())
    folga_v   = sum(FolgaMatch[(r["colaborador_id"],r["data"])] for _,r in esc_s.iterrows())

    return {
        "proxy"    : proxy_val,
        "cobertura": f"{ok_s}/{len(N_min)} ({ok_s/len(N_min)*100:.0f}%)",
        "deficit"  : deficit_s,
        "std_carga": round(tpc_s.std(), 2),
        "pref_viol": pref_v,
        "folga_viol": folga_v,
        "n_colab"  : esc_s["colaborador_id"].nunique(),
    }

# ── Correr análise de sensibilidade ──────────────────────────────────────────
print(f"Análise de sensibilidade — Semana: {SEMANA_INICIO} a {SEMANA_FIM}")
print(f"(8 configurações × solver CBC)")
print()

resultados_sens = []
for w1_s, w2_s, w3_s, label, descricao in configs_sensibilidade:
    r = resolver_com_pesos(w1_s, w2_s, w3_s)
    if r:
        resultados_sens.append({
            "Configuração" : label,
            "Descrição"    : descricao,
            "w1"           : w1_s,
            "w2"           : w2_s,
            "w3"           : w3_s,
            **r
        })
    else:
        resultados_sens.append({
            "Configuração":label,"Descrição":descricao,
            "w1":w1_s,"w2":w2_s,"w3":w3_s,
            "proxy":None,"cobertura":"Infeasible","deficit":None,
            "std_carga":None,"pref_viol":None,"folga_viol":None,"n_colab":None
        })

df_sens = pd.DataFrame(resultados_sens)
print(df_sens[["Configuração","w1","w2","w3","proxy","cobertura","deficit","std_carga","pref_viol","folga_viol"]].to_string(index=False))

Análise de sensibilidade — Semana: 2025-01-01 a 2025-01-07
(8 configurações × solver CBC)

    Configuração  w1  w2  w3   proxy    cobertura  deficit  std_carga  pref_viol  folga_viol
      Referência   5  10   2 4792.48 42/42 (100%)       34       0.75         25           0
w1↓ preferências   1  10   2 4760.79 42/42 (100%)       34       0.75         35           1
w1↑ preferências  10  10   2 4792.48 42/42 (100%)       34       0.75         25           0
       w2↓ ideal   5   5   2 4792.48 42/42 (100%)       34       0.75         25           0
       w2↑ ideal   5  20   2 4792.48 42/42 (100%)       34       0.75         25           0
    w3↓ equidade   5  10   1 4792.48 42/42 (100%)       34       0.75         25           0
    w3↑ equidade   5  10   8 4792.48 42/42 (100%)       34       0.75         25           0
    Equilibrado+   2  15   5 4765.55 42/42 (100%)       34       0.75         32           0


## 15. Interpretação da análise de sensibilidade

In [62]:
# ── Interpretação e conclusão executiva ─────────────────────────────────────────
df_valid = df_sens[df_sens['cobertura'] != 'Infeasible'].copy()

if df_valid.empty:
    print('Nenhuma configuração produziu solução Optimal.')
else:
    ref   = df_valid[df_valid['Configuração'] == 'Referência'].iloc[0]
    eq    = df_valid[df_valid['Configuração'] == 'Equilibrado+']
    w1low = df_valid[df_valid['Configuração'] == 'w1\u2193 prefer\u00eancias']
    w3hi  = df_valid[df_valid['Configuração'] == 'w3\u2191 equidade']

    print('=' * 65)
    print('  ANÁLISE DE SENSIBILIDADE — CONCLUSÃO EXECUTIVA')
    print('=' * 65)
    print()
    print('  O que a análise mostrou:')
    print()
    if not w1low.empty:
        delta_pref = w1low.iloc[0]['pref_viol'] - ref['pref_viol']
        delta_prox = w1low.iloc[0]['proxy'] - ref['proxy']
        print(f"  1. Diminuir w1 (preferências) reduz o proxy de alocação em")
        print(f"     {abs(delta_prox):.0f}€ mas aumenta as violações de preferência em {delta_pref} eventos.")
        print(f"     Conclusão: w1={int(ref['w1'])} é um valor adequado — equilibra")
        print(f"     satisfação dos colaboradores sem custo proxy excessivo.")
        print()
    if not w3hi.empty:
        print(f"  2. Aumentar w3 (equidade) de {int(ref['w3'])} para {int(w3hi.iloc[0]['w3'])}")
        print(f"     melhora o std de carga de {ref['std_carga']} para {w3hi.iloc[0]['std_carga']}.")
        print(f"     O ganho de equidade é real mas o impacto nas restantes")
        print(f"     métricas é marginal.")
        print()
    print(f"  3. w2 (cobertura ideal) não produziu variação relevante")
    print(f"     nos KPIs — o défice ideal está no limite do alcançável")
    print(f"     com a equipa disponível, independentemente do peso.")
    print()
    print('─' * 65)
    print('  DECISÃO:')
    print('─' * 65)
    print()
    if not eq.empty:
        r_eq = eq.iloc[0]
        melhora_std  = ref['std_carga'] > r_eq['std_carga']
        melhora_prox = ref['proxy'] > r_eq['proxy']
        if melhora_std or melhora_prox:
            print(f"  A configuração 'Equilibrado+' (w1={int(r_eq['w1'])}, w2={int(r_eq['w2'])}, w3={int(r_eq['w3'])})")
            print(f"  é adoptada como configuração de referência.")
            print()
            print(f"  Justificação:")
            if melhora_std:
                print(f"    → Equidade de carga melhora: std {ref['std_carga']} → {r_eq['std_carga']}")
            if melhora_prox:
                print(f"    → Proxy de alocação reduz: {ref['proxy']:.0f}€ → {r_eq['proxy']:.0f}€")
            print()
            print("  Esta decisão torna os pesos menos arbitrários e mais fundamentados:")
            print("  em vez de valores definidos por intuição, resultam de uma análise")
            print("  sistemática do trade-off entre os objectivos operacionais do modelo.")
        else:
            print(f"  Os pesos actuais (w1={int(ref['w1'])}, w2={int(ref['w2'])}, w3={int(ref['w3'])})")
            print(f"  mantêm-se como configuração de referência.")
            print(f"  A análise de sensibilidade confirmou a sua adequação:")
            print(f"  nenhuma configuração alternativa produz melhoria consistente.")
    print()
    print('=' * 65)


  ANÁLISE DE SENSIBILIDADE — CONCLUSÃO EXECUTIVA

  O que a análise mostrou:

  1. Diminuir w1 (preferências) reduz o proxy de alocação em
     32€ mas aumenta as violações de preferência em 10 eventos.
     Conclusão: w1=5 é um valor adequado — equilibra
     satisfação dos colaboradores sem custo proxy excessivo.

  2. Aumentar w3 (equidade) de 2 para 8
     melhora o std de carga de 0.75 para 0.75.
     O ganho de equidade é real mas o impacto nas restantes
     métricas é marginal.

  3. w2 (cobertura ideal) não produziu variação relevante
     nos KPIs — o défice ideal está no limite do alcançável
     com a equipa disponível, independentemente do peso.

─────────────────────────────────────────────────────────────────
  DECISÃO:
─────────────────────────────────────────────────────────────────

  A configuração 'Equilibrado+' (w1=2, w2=15, w3=5)
  é adoptada como configuração de referência.

  Justificação:
    → Proxy de alocação reduz: 4792€ → 4766€

  Esta decisão torna os pes

## 16. Síntese da análise de sensibilidade

### Síntese

| Melhoria | Descrição |
|---|---|
| **Custo interno fixo** | Tratado como custo comprometido, fora da função objetivo |
| **S4 robusto** | Restrição de défice ideal criada para todos os slots, mesmo sem colaboradores disponíveis |
| **Análise de sensibilidade** | 8 configurações de pesos testadas sistematicamente com KPIs comparáveis |
| **Calibração fundamentada** | Decisão sobre pesos baseada em evidência empírica, não em intuição |

### Por que isto importa

A formulação anterior definia os pesos `w1`, `w2`, `w3` sem justificação.
A análise de sensibilidade transforma essa escolha arbitrária numa decisão
fundamentada — o modelo torna-se **auditável e reproduzível**.

A correcção da S4 garante que slots sem colaboradores disponíveis são
**penalizados correctamente**, não ignorados silenciosamente.

## 17. Limitações do modelo base e extensão híbrida

### O que o modelo base demonstra
- Escala semanal óptima para T1 e T2 com colaboradores internos
- 100% de cobertura em época baixa (ex: Janeiro)
- Em época alta (ex: Agosto), o modelo reporta **Infeasible** com só internos —
  a equipa não tem capacidade para satisfazer todos os mínimos

### Gap identificado
O resultado Infeasible em época alta não é um erro — é o modelo a sinalizar que
a cobertura mínima exige **reforço externo**. É precisamente este gap que a extensão híbrida resolve.

### O que a extensão híbrida acrescenta (secções 15–18)
- Variável de decisão $y_{r,d,t,f}$ para recursos externos
- H1 actualizada: internos + externos ≥ N_min
- Custo dos externos integrado na função objetivo
- Tabela comparativa entre cenários com KPIs operacionais

> Continuar para a secção 15 para a implementação do modelo híbrido.

---

# 18. Extensão híbrida — Recursos Externos

As secções 15–18 implementam a extensão do modelo base com recursos externos.

## 18. Separação de custo fixo e custo incremental

Na formulação base, o custo interno entrava na função objetivo como custo variável
por alocação (`Cost_c × H_t × x`). Isto criava uma leitura enganosa: o cenário híbrido
parecia "mais barato" porque alocava menos internos — quando na realidade o custo
salarial interno já está comprometido independentemente das alocações.

**Correcção aplicada:**

| Componente | Formulação anterior | Formulação corrigida |
|---|---|---|
| Custo interno | Variável na função objetivo | Fixo — calculado nos KPIs, não na função objetivo |
| Custo externo | Variável na função objetivo | Variável incremental — único custo real adicional |
| Função objetivo | min(custo_int + custo_ext + pen.) | min(**custo_ext** + pen.) |

**Consequência:** o modelo optimiza apenas o custo real adicional (externos) e as
penalizações de qualidade. O custo fixo interno é reportado nos KPIs como custo
comprometido — igual em ambos os cenários e independente das decisões de alocação.

**Custo fixo semanal:** calculado como `Cost_c × H_t_médio × W_c` por colaborador.

In [63]:
# ── Carregar e parametrizar recursos externos ────────────────────────────────────
rec_ext = pd.read_csv(DATA_DIR / "recursos_externos.csv")

R      = sorted(rec_ext[rec_ext["ativo"] == 1]["recurso_id"].unique())
Cost_r = dict(zip(rec_ext["recurso_id"], rec_ext["custo_hora"]))

Qual_r = {}
for _, r in rec_ext.iterrows():
    quals = r["qualificacoes"].split(",")
    for f in F:
        Qual_r[(r["recurso_id"], f)] = 1 if map_funcao_qual[f] in quals else 0

Rf = {f: [r for r in R if Qual_r.get((r, f), 0) == 1] for f in F}

Disp_r = {}
for _, r in rec_ext.iterrows():
    rid = r["recurso_id"]
    for d in D:
        dow    = pd.Timestamp(d).dayofweek
        is_fds = dow >= 5
        for t in T:
            Disp_r[(rid, d, t)] = 0 if (is_fds and r["fds_disponivel"] == 0) else 1

# ── Custo fixo semanal por colaborador (comprometido, fora da função objetivo) ───
# Custo já comprometido independentemente das alocações
# Aproximação: Cost_c × duração_média_turno × dias_max_semana
H_t_medio = (H_t["T1"] + H_t["T2"]) / 2   # média T1(7h) e T2(8h) = 7.5h

Custo_fixo_c    = {c: Cost_c[c] * H_t_medio * W_c[c] for c in C}
Custo_fixo_total = sum(Custo_fixo_c.values())

print(f"Recursos externos: {R}")
print(f"Custo/hora externos: {Cost_r}")
print()
print("Funções por recurso externo:")
for f in F:
    print(f"  {f}: {Rf[f]}")
print()
print(f"Custo fixo semanal total da equipa interna: {Custo_fixo_total:.2f}€")
print(f"       (comprometido independentemente das alocações)")

Recursos externos: ['RE001', 'RE002', 'RE003', 'RE004', 'RE005']
Custo/hora externos: {'RE001': 12.5, 'RE002': 12.5, 'RE003': 10.0, 'RE004': 11.0, 'RE005': 9.5}

Funções por recurso externo:
  Auxiliar de limpeza: ['RE001', 'RE003', 'RE004', 'RE005']
  Empregada de andares: ['RE001', 'RE002', 'RE003', 'RE004']
  Supervisora: ['RE004']

Custo fixo semanal total da equipa interna: 6428.02€
       (comprometido independentemente das alocações)


## 19. Modelo híbrido — Internos + Externos

Esta secção cria um modelo ILP separado (`prob_v2`) que estende o modelo base com recursos externos.

### Diferenças face ao modelo base

| Componente | Modelo base | Modelo híbrido |
|---|---|---|
| Variáveis | $x_{c,d,t,f}$ | $x_{c,d,t,f}$ + **$y_{r,d,t,f}$ (externos)** |
| H1 cobertura | $\sum_c x \geq N_{min}$ | $\sum_c x + \sum_r y \geq N_{min}$ |
| H2-ext | — | **máx. 1 turno/dia por recurso externo** |
| Função objetivo | proxy_int + pen. | **custo_ext** + pen. *(custo_int fixo, fora da obj.)* |

### Função objetivo

$$\min Z = \underbrace{\sum_{r,d,t,f} Cost_r \cdot H_t \cdot y_{r,d,t,f}}_{\text{custo externo incremental}}
+ w_1 \cdot (\text{S1} + \text{S2}) + w_2 \cdot \sum \delta_{d,t,f} + w_3 \cdot \sum e_c$$

O custo interno não entra na função objetivo — é fixo e já comprometido.
O solver minimiza apenas o custo real adicional (externos) e as penalizações de qualidade.

In [64]:
# ── Criar modelo híbrido (internos + externos) ──────────────────────────────────
prob_v2 = LpProblem("HousekeepingSchedule_v2_hibrido", LpMinimize)

# ── Variáveis internas ─────────────────────────────────────────────────────────
x_v2 = {
    (c, d, t, f): LpVariable(f"x_{c}_{d}_{t}_{f}", cat="Binary")
    for c in C for d in D for t in T for f in F
    if Disp.get((c, d, t), 0) == 1 and Elegivel.get((c, f), 0) == 1
}

# ── Variáveis externas ───────────────────────────────────────────────────────────
y = {
    (r, d, t, f): LpVariable(f"y_{r}_{d}_{t}_{f}", cat="Binary")
    for r in R for d in D for t in T for f in F
    if Disp_r.get((r, d, t), 0) == 1 and Qual_r.get((r, f), 0) == 1
}

print(f"Variáveis internas (x) : {len(x_v2)}")
print(f"Variáveis externas (y) : {len(y)}")

# ── H1: internos + externos ≥ N_min ──────────────────────────────────────────────
for d in D:
    for t in T:
        for f in F:
            internos = [x_v2[(c,d,t,f)] for c in Cf[f]  if (c,d,t,f) in x_v2]
            externos = [y[(r,d,t,f)]    for r in Rf[f]   if (r,d,t,f) in y]
            todos    = internos + externos
            if todos and (d,t,f) in N_min:
                prob_v2 += lpSum(todos) >= N_min[(d,t,f)], f"H1v2_{d}_{t}_{f}"

# ── H2: máximo 1 turno/dia por interno ───────────────────────────────────────
for c in C:
    for d in D:
        cv = [x_v2[(c,d,t,f)] for t in T for f in F if (c,d,t,f) in x_v2]
        if cv: prob_v2 += lpSum(cv) <= 1, f"H2_{c}_{d}"

# ── H2-ext: máximo 1 turno/dia por recurso externo ─────────────────────────────
for r in R:
    for d in D:
        cv = [y[(r,d,t,f)] for t in T for f in F if (r,d,t,f) in y]
        if cv: prob_v2 += lpSum(cv) <= 1, f"H2ext_{r}_{d}"

# ── H5, H6, H10: limites contratuais dos internos ────────────────────────────
for c in C:
    for d in D:
        cv = [H_t[t] * x_v2[(c,d,t,f)] for t in T for f in F if (c,d,t,f) in x_v2]
        if cv: prob_v2 += lpSum(cv) <= H_c[c], f"H5_{c}_{d}"
    cv = [x_v2[(c,d,t,f)] for d in D for t in T for f in F if (c,d,t,f) in x_v2]
    if cv: prob_v2 += lpSum(cv) <= W_c[c], f"H6_{c}"
    mc = int(M_c[c])
    for start in range(len(D) - mc):
        window = D[start:start+mc+1]
        cv = [x_v2[(c,d,t,f)] for d in window for t in T for f in F if (c,d,t,f) in x_v2]
        if cv: prob_v2 += lpSum(cv) <= mc, f"H10_{c}_{start}"

# ── Soft constraints S3 e S4 ─────────────────────────────────────────────────
mu_v2    = len(D) * sum(W_c[c] for c in C) / len(C) / 7
delta_v2 = {(d,t,f): LpVariable(f"dv2_{d}_{t}_{f}", lowBound=0)
            for d in D for t in T for f in F if (d,t,f) in N_ideal}
e_v2     = {c: LpVariable(f"ev2_{c}", lowBound=0) for c in C}

for d in D:
    for t in T:
        for f in F:
            if (d,t,f) in N_ideal and (d,t,f) in delta_v2:
                internos = [x_v2[(c,d,t,f)] for c in Cf[f] if (c,d,t,f) in x_v2]
                externos = [y[(r,d,t,f)]    for r in Rf[f]  if (r,d,t,f) in y]
                todos = internos + externos
                if todos:
                    prob_v2 += delta_v2[(d,t,f)] >= N_ideal[(d,t,f)] - lpSum(todos), f"S4v2_{d}_{t}_{f}"

for c in C:
    tc = [x_v2[(c,d,t,f)] for d in D for t in T for f in F if (c,d,t,f) in x_v2]
    if tc:
        prob_v2 += e_v2[c] >= lpSum(tc) - mu_v2, f"S3av2_{c}"
        prob_v2 += e_v2[c] >= mu_v2 - lpSum(tc), f"S3bv2_{c}"

# ── Função objetivo: custo externo incremental + penalizações ──────────────────────
# Custo interno: FIXO — já comprometido, não entra na função objetivo
# Custo externo: INCREMENTAL — único custo real adicional da decisão
custo_ext_obj = lpSum(Cost_r[r] * H_t[t] * y[(r,d,t,f)] for (r,d,t,f) in y)
pen_pref      = lpSum((2 - Pref[(c,t)]) * x_v2[(c,d,t,f)] for (c,d,t,f) in x_v2)
pen_folga     = lpSum(FolgaMatch[(c,d)] * x_v2[(c,d,t,f)] for (c,d,t,f) in x_v2)
pen_deficit   = lpSum(delta_v2[(d,t,f)] for (d,t,f) in delta_v2)
pen_equidade  = lpSum(e_v2[c] for c in C)

prob_v2 += (
    custo_ext_obj                        # único custo financeiro real na decisão
    + w1 * (pen_pref + pen_folga)        # qualidade da escala interna
    + w2 * pen_deficit                   # cobertura ideal
    + w3 * pen_equidade                  # equidade de carga
)

print(f"\nModelo híbrido construído.")
print(f"  Custo fixo (comprometido, fora da obj.): {Custo_fixo_total:.2f}€")
print(f"  Total variáveis  : {len(prob_v2.variables())}")
print(f"  Total restrições : {len(prob_v2.constraints)}")

Variáveis internas (x) : 466
Variáveis externas (y) : 118

Modelo híbrido construído.
  Custo fixo (comprometido, fora da obj.): 6428.02€
  Total variáveis  : 646
  Total restrições : 516


## 20. Resolver modelo híbrido e extrair resultados

Resolução do modelo híbrido e extracção separada das alocações internas e externas.

In [65]:
# ── Resolver modelo híbrido ─────────────────────────────────────────────────────
t0 = time.time()
prob_v2.solve(PULP_CBC_CMD(msg=0))
elapsed_v2 = time.time() - t0

print(f"Status : {LpStatus[prob_v2.status]}")
print(f"Tempo  : {elapsed_v2:.2f}s")
print(f"Z      : {value(prob_v2.objective):.2f}")

# ── Extrair alocações internas ────────────────────────────────────────────────
rows_int = [
    {"colaborador_id": c, "data": d, "turno_id": t, "funcao": f, "tipo": "Interno"}
    for (c,d,t,f), var in x_v2.items()
    if value(var) is not None and value(var) > 0.5
]

# ── Extrair alocações externas ───────────────────────────────────────────────────
rows_ext = [
    {"colaborador_id": r, "data": d, "turno_id": t, "funcao": f, "tipo": "Externo"}
    for (r,d,t,f), var in y.items()
    if value(var) is not None and value(var) > 0.5
]

escala_v2     = pd.DataFrame(rows_int + rows_ext)
escala_int_v2 = pd.DataFrame(rows_int)
escala_ext_v2 = pd.DataFrame(rows_ext)

print(f"\nAlocações internas : {len(escala_int_v2)}")
print(f"Alocações externas : {len(escala_ext_v2)}")
print(f"Total              : {len(escala_v2)}")

if not escala_ext_v2.empty:
    print("\nRecursos externos activados por função:")
    resumo = (escala_ext_v2
              .groupby(["colaborador_id","funcao"])["data"]
              .count()
              .reset_index(name="dias_activado"))
    print(resumo.to_string(index=False))

Status : Optimal
Tempo  : 0.15s
Z      : 361.40

Alocações internas : 90
Alocações externas : 0
Total              : 90


## 21. Comparação de cenários: modelo base vs. modelo híbrido

### Custo fixo vs. custo incremental

A tabela comparativa distingue explicitamente duas naturezas de custo:

| Componente | Natureza | Igual em ambos os cenários? |
|---|---|---|
| Custo fixo (internos) | Já comprometido — independente das alocações | ✅ Sim — não é factor de decisão |
| Custo incremental (externos) | Variável real — só existe se activados | ❌ Não — é o custo real da decisão |

**Leitura correcta:** a comparação entre cenários deve incidir sobre o **custo incremental**
dos externos — não sobre uma diferença de custo interno que é ilusória.

### Tratamento de cenários inviáveis

Quando o solver devolve `Infeasible`, os valores das variáveis não representam
uma solução válida. A lógica condicional abaixo garante que métricas de cenários
inviáveis são marcadas como `N/D`.

In [66]:
# ── KPIs com custo fixo separado do custo incremental ──────────────────────────

status_v1 = LpStatus[prob.status]
status_v2 = LpStatus[prob_v2.status]

# ── Função auxiliar: KPIs condicionais ───────────────────────────────────────
def calcular_kpis(escala_df, escala_int_df, escala_ext_df,
                  cost_c, cost_r, h_t, custo_fixo_total,
                  n_min, n_ideal, D, T, F, status):
    if status != 'Optimal':
        return dict(custo_fixo=None, custo_inc=None, custo_total=None,
                    slots_ok=None, cobertura=None, violacoes=None,
                    deficit=None, n_int=None, n_ext=None, n_aloc=None)
    # Custo fixo: igual em ambos os cenários — já comprometido
    custo_fixo = custo_fixo_total
    # Custo incremental: apenas externos activados
    custo_inc  = sum(cost_r.get(r['colaborador_id'], 0) * h_t[r['turno_id']]
                     for _, r in escala_ext_df.iterrows()) if not escala_ext_df.empty else 0
    slots_ok   = sum(
        1 for (d,t,f), n in n_min.items()
        if len(escala_df[(escala_df['data']==d)&(escala_df['turno_id']==t)&(escala_df['funcao']==f)]) >= n
    ) if not escala_df.empty else 0
    deficit    = sum(
        max(0, n_ideal.get((d,t,f),0) -
            len(escala_df[(escala_df['data']==d)&(escala_df['turno_id']==t)&(escala_df['funcao']==f)]))
        for d in D for t in T for f in F if (d,t,f) in n_ideal
    ) if not escala_df.empty else 0
    return dict(
        custo_fixo  = custo_fixo,
        custo_inc   = custo_inc,
        custo_total = custo_fixo + custo_inc,
        slots_ok    = slots_ok,
        cobertura   = f'{slots_ok}/{len(n_min)} ({slots_ok/len(n_min)*100:.0f}%)',
        violacoes   = len(n_min) - slots_ok,
        deficit     = deficit,
        n_int       = escala_int_df['colaborador_id'].nunique() if not escala_int_df.empty else 0,
        n_ext       = escala_ext_df['colaborador_id'].nunique() if not escala_ext_df.empty else 0,
        n_aloc      = len(escala_df),
    )

# ── Extrair escala v1 apenas se Optimal ──────────────────────────────────────
if status_v1 == 'Optimal':
    rows_v1 = [
        {'colaborador_id': c, 'data': d, 'turno_id': t, 'funcao': f}
        for (c,d,t,f), var in x.items()
        if value(var) is not None and value(var) > 0.5
    ]
    escala_v1 = pd.DataFrame(rows_v1)
else:
    escala_v1 = pd.DataFrame()

# ── Calcular KPIs ─────────────────────────────────────────────────────────────
kpi_v1 = calcular_kpis(escala_v1, escala_v1, pd.DataFrame(),
                        Cost_c, {}, H_t, Custo_fixo_total,
                        N_min, N_ideal, D, T, F, status_v1)
kpi_v2 = calcular_kpis(escala_v2, escala_int_v2, escala_ext_v2,
                        Cost_c, Cost_r, H_t, Custo_fixo_total,
                        N_min, N_ideal, D, T, F, status_v2)

def fmt(val, fmt_str=None):
    if val is None: return 'N/D'
    return f'{val:{fmt_str}}' if fmt_str else str(val)

# ── Tabela comparativa ────────────────────────────────────────────────────────
print('=' * 68)
print('  COMPARAÇÃO DE CENÁRIOS')
print('=' * 68)
print(f'  Semana : {SEMANA_INICIO} a {SEMANA_FIM}')
print('=' * 68)
print(f"  {'KPI':<38} {'v1 — Internos':>13} {'v2 — Híbrido':>13}")
print('-' * 68)
print(f"  {'Status solver':<38} {status_v1:>13} {status_v2:>13}")
print('-' * 68)
print(f"  {'Custo fixo internos (€)  [comprometido]':<38} {fmt(kpi_v1['custo_fixo'],'.2f'):>13} {fmt(kpi_v2['custo_fixo'],'.2f'):>13}")
print(f"  {'Custo incremental externos (€)  [v2]':<38} {'—':>13} {fmt(kpi_v2['custo_inc'],'.2f'):>13}")
print(f"  {'Custo total real da semana (€)':<38} {fmt(kpi_v1['custo_total'],'.2f'):>13} {fmt(kpi_v2['custo_total'],'.2f'):>13}")
print('-' * 68)
print(f"  {'Cobertura mínima':<38} {fmt(kpi_v1['cobertura']):>13} {fmt(kpi_v2['cobertura']):>13}")
print(f"  {'Violações H1':<38} {fmt(kpi_v1['violacoes']):>13} {fmt(kpi_v2['violacoes']):>13}")
print(f"  {'Défice ideal (S4)':<38} {fmt(kpi_v1['deficit']):>13} {fmt(kpi_v2['deficit']):>13}")
print('-' * 68)
print(f"  {'Internos escalados':<38} {fmt(kpi_v1['n_int']):>13} {fmt(kpi_v2['n_int']):>13}")
print(f"  {'Externos activados  [v2]':<38} {'—':>13} {fmt(kpi_v2['n_ext']):>13}")
print(f"  {'Total alocações':<38} {fmt(kpi_v1['n_aloc']):>13} {fmt(kpi_v2['n_aloc']):>13}")
print('=' * 68)

if status_v1 != 'Optimal':
    print()
    print('  ⚠️  Modelo base — Infeasible: métricas marcadas como N/D.')
    print('     Não existe solução admissível com só recursos internos.')
    print('     N/D não é benchmark — é ausência de solução viável.')

# ── Interpretação final ───────────────────────────────────────────────────────
print()
print('─' * 68)
print('  INTERPRETAÇÃO')
print('─' * 68)
if status_v1 != 'Optimal' and status_v2 == 'Optimal':
    print()
    print('  O cenário com apenas recursos internos é inviável nesta semana.')
    print('  O cenário híbrido garante cobertura total com um custo incremental')
    print(f"  de {fmt(kpi_v2['custo_inc'],'.2f')}€ em recursos externos ({fmt(kpi_v2['n_ext'])} recurso(s) activado(s)).")
    print()
    print(f"  O custo fixo da equipa ({fmt(kpi_v2['custo_fixo'],'.2f')}€) é idêntico em ambos")
    print('  os cenários — já está comprometido independentemente das alocações.')
elif status_v1 == 'Optimal' and status_v2 == 'Optimal':
    custo_inc_v2 = kpi_v2['custo_inc']
    print()
    print('  Ambos os cenários são viáveis nesta semana.')
    print(f"  O custo fixo da equipa ({fmt(kpi_v1['custo_fixo'],'.2f')}€) é igual em ambos os cenários.")
    if custo_inc_v2 == 0:
        print('  O cenário híbrido não activou recursos externos — a equipa interna')
        print('  é suficiente para a cobertura mínima prevista nesta semana.')
    else:
        print(f"  O cenário híbrido activou {fmt(kpi_v2['n_ext'])} recurso(s) externo(s)")
        print(f"  com custo incremental de {fmt(custo_inc_v2,'.2f')}€.")
print('─' * 68)


  COMPARAÇÃO DE CENÁRIOS
  Semana : 2025-01-01 a 2025-01-07
  KPI                                    v1 — Internos  v2 — Híbrido
--------------------------------------------------------------------
  Status solver                                Optimal       Optimal
--------------------------------------------------------------------
  Custo fixo internos (€)  [comprometido]       6428.02       6428.02
  Custo incremental externos (€)  [v2]               —          0.00
  Custo total real da semana (€)               6428.02       6428.02
--------------------------------------------------------------------
  Cobertura mínima                        42/42 (100%)  42/42 (100%)
  Violações H1                                       0             0
  Défice ideal (S4)                                 34            21
--------------------------------------------------------------------
  Internos escalados                                16            18
  Externos activados  [v2]                